# 🏠 House Price Prediction (Regression)
**Target:** `medv` — Median house value in $1000s

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 2. Load Dataset

In [ ]:
try:
    df = pd.read_csv('BostonHousing.csv')
except FileNotFoundError:
    print("CSV not found."); raise
print(f"Shape: {df.shape}"); df.head()

## 3. Identify Data Types

In [ ]:
print("Data types:"); print(df.dtypes)
print(f"\nNumeric : {df.select_dtypes(include='number').columns.tolist()}")
print(f"Object  : {df.select_dtypes(include='object').columns.tolist()}")

## 4. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 5. Handle Missing Values

In [ ]:
print("Missing:"); print(df.isnull().sum())
for col in df.columns:
    if df[col].isnull().sum()>0:
        med=df[col].median(); df[col].fillna(med,inplace=True)
        print(f"  '{col}' filled with median ({med:.3f})")
print(f"Remaining: {df.isnull().sum().sum()}")

## 6. Handle Duplicates

In [ ]:
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True); print(f"Shape: {df.shape}")

## 7. Outlier Detection & Handling

In [ ]:
num_cols=df.select_dtypes(include='number').columns.tolist()
fig,axes=plt.subplots(2,7,figsize=(18,6))
for ax,col in zip(axes.flatten(),num_cols):
    ax.boxplot(df[col],vert=True,patch_artist=True,boxprops=dict(facecolor='steelblue',alpha=0.6))
    ax.set_title(col,fontsize=8); ax.set_xticks([])
for ax in axes.flatten()[len(num_cols):]: ax.set_visible(False)
plt.suptitle('Boxplots',fontsize=12,y=1.02); plt.tight_layout(); plt.show()
before=len(df)
for col in ['crim','lstat','medv']:
    Q1,Q3=df[col].quantile([0.25,0.75]); IQR=Q3-Q1
    df=df[df[col].between(Q1-1.5*IQR,Q3+1.5*IQR)]
print(f"Removed: {before-len(df)} | Shape: {df.shape}")

## 8. Visualizations & Insights

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].hist(df['medv'],bins=25,color='steelblue',edgecolor='white')
axes[0].axvline(df['medv'].mean(),color='red',linestyle='--',label=f"Mean: {df['medv'].mean():.1f}")
axes[0].set_title('House Price Distribution'); axes[0].legend()

axes[1].scatter(df['rm'],df['medv'],alpha=0.5,color='coral',s=20)
axes[1].set_title('Rooms vs Price'); axes[1].set_xlabel('Avg Rooms'); axes[1].set_ylabel('medv')

corr=df.corr()['medv'].drop('medv').sort_values()
colors_bar=['tomato' if v<0 else 'steelblue' for v in corr]
axes[2].barh(corr.index,corr.values,color=colors_bar)
axes[2].axvline(0,color='black',linewidth=0.8); axes[2].set_title('Correlation with medv')

plt.tight_layout(); plt.show()
print("""Insights:
1. Price slightly right-skewed; most houses $15k–$30k.
2. Strong positive correlation between rooms (rm) and price.
3. lstat (poverty %) is the strongest negative predictor.""")

## 9. Feature Selection & Scaling

In [ ]:
threshold=0.3
selected=df.corr()['medv'].drop('medv')
selected=selected[selected.abs()>threshold].index.tolist()
print(f"Selected features: {selected}")
X=df[selected]; y=df['medv']
FEATURE_COLS=selected

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train); X_test_sc=scaler.transform(X_test)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 10. Model Building

In [ ]:
models={
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=100,max_depth=8,random_state=42,n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42)
}
results={}
for name,model in models.items():
    model.fit(X_train_sc,y_train); yp=model.predict(X_test_sc)
    results[name]={'RMSE':round(np.sqrt(mean_squared_error(y_test,yp)),3),
                   'MAE' :round(mean_absolute_error(y_test,yp),3),
                   'R²'  :round(r2_score(y_test,yp),4)}
    print(f'✓ {name}')

## 11. Model Comparison

In [ ]:
res_df=pd.DataFrame(results).T.sort_values('R²',ascending=False); print(res_df)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,metric,color in zip(axes,['RMSE','MAE','R²'],['#e05c5c','#e09b5c','#5c9ee0']):
    vals=res_df[metric]; bars=ax.bar(vals.index,vals.values,color=color,edgecolor='white',width=0.5)
    ax.set_title(metric,fontweight='bold'); ax.set_xticklabels(vals.index,rotation=15,ha='right',fontsize=8)
    for b,v in zip(bars,vals.values): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f'{v:.3f}',ha='center',fontsize=8)
plt.suptitle('Model Comparison',fontsize=13,y=1.02); plt.tight_layout(); plt.show()

---
## 🔮 12. Predict House Price for Your Own Input
**Edit the values below and run the cell.**

In [ ]:
# ╔══════════════════════════════════════════╗
# ║   ✏️  CHANGE THESE VALUES TO YOUR INPUT  ║
# ╚══════════════════════════════════════════╝
# Only features selected during training are needed.
# Check FEATURE_COLS printed in Step 9 and fill accordingly.

input_values = {
    'rm'      : 6.5,    # Avg number of rooms per dwelling (typical: 4–9)
    'lstat'   : 10.0,   # % lower status population (typical: 2–38)
    'ptratio' : 15.0,   # Pupil-teacher ratio (typical: 12–22)
    'indus'   : 5.0,    # % non-retail business acres (typical: 0.5–27)
    'nox'     : 0.45,   # Nitric oxide concentration (typical: 0.38–0.87)
    'crim'    : 0.05,   # Per capita crime rate (typical: 0–89)
    'age'     : 50.0,   # % units built before 1940 (typical: 3–100)
    'dis'     : 4.0,    # Distance to employment centres (typical: 1–12)
    'tax'     : 300.0,  # Property tax rate (typical: 187–711)
    'b'       : 390.0,  # 1000*(Bk-0.63)^2 (typical: 0–397)
    'rad'     : 4,      # Highway accessibility index (typical: 1–24)
    'zn'      : 0.0,    # % residential land for large lots (typical: 0–100)
    'chas'    : 0,      # Charles River (0=No, 1=Yes)
}

# ── Auto-process ─────────────────────────
new_input = pd.DataFrame([input_values])
new_input = new_input.reindex(columns=FEATURE_COLS, fill_value=0)
new_scaled = scaler.transform(new_input)

print("=" * 45)
print("     🏠 HOUSE PRICE PREDICTION RESULTS")
print("=" * 45)
for k, v in input_values.items():
    if k in FEATURE_COLS:
        print(f"  {k:<10}: {v}")
print("-" * 45)
for name, model in models.items():
    pred = model.predict(new_scaled)[0]
    print(f"  {name:<22}: ${pred*1000:,.0f}  (${pred:.1f}k)")
print("=" * 45)